# Практика: датаклассы

На этом занятии:

- выясним, почему обычный класс, предназначенный для хранения данных, требует много однотипного кода, и как этот код убирает декоратор `dataclass`;
- рассмотрим параметры декоратора `frozen`, `kw_only`, `order` и `init`;
- убедимся, что датакласс остается обычным классом, и добавим к нему собственные методы;
- создадим атрибуты класса с помощью `ClassVar`;
- познакомимся со `__slots__` и параметром `slots=True`;
- изучим функцию `field`, значения по умолчанию (в том числе для изменяемых типов) и `default_factory`;
- проверим входные данные в методе `__post_init__`;
- преобразуем датакласс в словарь и кортеж функциями `asdict` и `astuple`.

Все темы разбираем на одном примере: запись о пользователе интернет-магазина с полями `id`, `username`, `password`, `email` и `created_at`.

In [ ]:
import json
import time
import tracemalloc
from dataclasses import asdict, astuple, dataclass, field
from datetime import UTC, datetime
from typing import Any, ClassVar

## 1. Класс для хранения данных без `dataclass`

Запись о пользователе — это набор значений: у нее есть состояние и почти нет поведения. Опишем такой класс обычным способом, как в [конспекте о методе `__init__`](conspects/07-instance-methods.md).

Значение `created_at` — объект `datetime` из модуля `datetime`: он хранит дату и время. Конструктор `datetime(2026, 11, 19, 10, 30)` принимает год, месяц, день, часы и минуты.

Имя поля `id` совпадает с именем встроенной функции `id()`. Внутри класса и метода это допустимо: имя лишь перекрывает встроенную функцию в этих областях видимости. Здесь используется `id`, как в постановке задачи; в рабочем коде часто выбирают `user_id`.

In [ ]:
class PlainUser:
    def __init__(
        self,
        id: int,
        username: str,
        password: str,
        email: str,
        created_at: datetime,
    ) -> None:
        self.id = id
        self.username = username
        self.password = password
        self.email = email
        self.created_at = created_at

### Проблема 1. Однотипный `__init__`

Метод `__init__` не содержит логики: он только присваивает значения параметров атрибутам. При этом каждое имя поля записывается три раза: в списке параметров (`id: int`), в имени атрибута (`self.id`) и в присваиваемом значении (`= id`). Добавление или переименование поля требует правок сразу в нескольких местах, а для классов с десятками полей такой код становится длинным и подвержен опечаткам.

### Проблема 2. Непонятное строковое представление

Создадим два экземпляра с одинаковыми значениями и напечатаем один из них.

In [ ]:
registered_at = datetime(2026, 11, 19, 10, 30)
first_user = PlainUser(1, "alice", "secret123", "alice@example.com", registered_at)
second_user = PlainUser(1, "alice", "secret123", "alice@example.com", registered_at)
print(first_user)  # <__main__.PlainUser object at 0x000001FBFA0B57F0>

Функция `print` выводит только имя класса и адрес объекта в памяти. Значения полей из такой строки узнать нельзя, а адрес меняется от запуска к запуску. Для отладки и логирования такое представление бесполезно.

### Проблема 3. Сравнение по идентичности

По умолчанию экземпляры пользовательского класса сравниваются по идентичности (по `id()`), а не по значениям атрибутов (см. [конспект об идентичности, типе и значении](../lesson02/conspects/05-objects-identity-type-value.md)).

In [ ]:
print(first_user == second_user)                    # False
print(first_user is second_user)                    # False
print(first_user.__dict__ == second_user.__dict__)  # True -- атрибуты совпадают

Две записи с одинаковыми данными считаются разными объектами. Для записи о пользователе, которая определяется своими значениями, это неудобно.

Исправить вторую и третью проблему вручную можно (как именно, разберем на следующем занятии), но это еще больше однотипного кода, который при каждом изменении набора полей нужно синхронизировать с `__init__`. Модуль `dataclasses` генерирует этот код автоматически.

## 2. Декоратор `dataclass`

Декоратор `dataclass` принимает класс, анализирует его аннотации и добавляет в класс сгенерированный `__init__`, а также код, который задает понятное строковое представление экземпляра и сравнение экземпляров по значениям. Поле датакласса объявляется **аннотацией** в теле класса: имя и тип. Порядок аннотаций определяет порядок параметров сгенерированного `__init__`.

In [ ]:
@dataclass
class User:
    id: int
    username: str
    password: str
    email: str
    created_at: datetime

In [ ]:
first_user = User(1, "alice", "secret123", "alice@example.com", registered_at)
second_user = User(1, "alice", "secret123", "alice@example.com", registered_at)
print(first_user)

Печать экземпляра теперь показывает имя класса и значения всех полей. Сравнение работает по значениям полей, при этом экземпляры остаются разными объектами:

In [ ]:
print(first_user == second_user)  # True -- значения полей совпадают
print(first_user is second_user)  # False -- это два разных объекта

Типы из аннотаций датакласс во время выполнения **не проверяет**: они нужны для документации и статических анализаторов (`mypy`, подсказки в IDE).

In [ ]:
broken_user = User(id="abc", username=42, password=None, email=[], created_at=0)
print(broken_user)

## 3. Параметры декоратора

Декоратор можно вызвать с аргументами: `@dataclass(frozen=True)`. Вызов `dataclass(...)` возвращает декоратор, который затем применяется к классу, то есть это [параметризованный декоратор](../lesson08/conspects/09-parametrized-decorators.md). Рассмотрим параметры `frozen`, `kw_only`, `order` и `init`.

### `frozen=True`: неизменяемые экземпляры

По умолчанию атрибуты экземпляра датакласса можно изменять. Параметр `frozen=True` запрещает после создания экземпляра присваивание атрибутов (в том числе повторную привязку имени к другому значению и создание новых атрибутов) и удаление атрибутов: такая попытка возбуждает `FrozenInstanceError` (подкласс `AttributeError`).

In [ ]:
@dataclass(frozen=True)
class FrozenUser:
    id: int
    username: str
    password: str
    email: str
    created_at: datetime


frozen_user = FrozenUser(1, "alice", "secret123", "alice@example.com", registered_at)

try:
    frozen_user.email = "new@example.com"

except AttributeError as error:
    print(f"{type(error).__name__}: {error}")

Неизменяемость связана с хешируемостью (см. [конспект о хешах](../lesson06/conspects/02-hash-and-hashability.md)). Обычный датакласс сравнивается по значениям и при этом изменяем, поэтому Python делает его **нехешируемым**. Датакласс с `frozen=True` становится хешируемым: хеш экземпляра вычисляется по значениям полей. Если значения всех полей хешируемы, экземпляры такого датакласса можно хранить во множествах и использовать как ключи словарей.

Параметр `frozen=True` сам по себе хешируемость экземпляра **не гарантирует**: он запрещает изменять набор и привязки атрибутов экземпляра, но не изменяемость объектов, на которые эти атрибуты ссылаются. Хеш экземпляра вычисляется как хеш кортежа значений полей, поэтому если хотя бы одно поле содержит нехешируемый объект (например, список), вызов `hash()` возбуждает `TypeError`, хотя сам экземпляр создается без ошибок.

In [ ]:
duplicate_user = FrozenUser(1, "alice", "secret123", "alice@example.com", registered_at)
print(len({frozen_user, duplicate_user}))  # 1 -- равные экземпляры
regular_user = User(1, "alice", "secret123", "alice@example.com", registered_at)

try:
    hash(regular_user)

except TypeError as error:
    print(error)  # unhashable type: 'User'

Параметр `frozen` защищает только сам экземпляр: привязки его атрибутов. Если атрибут ссылается на изменяемый объект (например, на список), сам этот объект изменить по-прежнему можно.

### `kw_only=True`: только именованные аргументы

Поля пользователя часто имеют один и тот же тип `str`. При позиционной передаче легко перепутать их порядок, и ошибка не будет обнаружена:

In [ ]:
mixed_user = User(1, "alice@example.com", "alice", "secret123", registered_at)
print(mixed_user.username)  # alice@example.com -- в username попал email

Параметр `kw_only=True` делает все параметры сгенерированного `__init__` **только именованными** (как параметры после `*` в [определении функции](../lesson07/conspects/06-positional-and-keyword-arguments.md)).

In [ ]:
@dataclass(kw_only=True)
class KeywordUser:
    id: int
    username: str
    password: str
    email: str
    created_at: datetime


try:
    KeywordUser(1, "alice", "secret123", "alice@example.com", registered_at)

except TypeError as error:
    print(error)

In [ ]:
keyword_user = KeywordUser(
    id=1,
    username="alice",
    password="secret123",
    email="alice@example.com",
    created_at=registered_at,
)
print(keyword_user.username)  # alice

Режим `kw_only` можно включить и для отдельного поля: `field(kw_only=True)` (функция `field` рассматривается в разделе 7).

### `order=True`: сравнение экземпляров

По умолчанию операторы `<`, `>`, `<=`, `>=` для датакласса не определены:

In [ ]:
try:
    print(regular_user < regular_user)

except TypeError as error:
    print(error)

Параметр `order=True` добавляет в класс поддержку операторов `<`, `<=`, `>` и `>=`. Экземпляры сравниваются как **кортежи значений полей в порядке их объявления**: сначала по `id`, при равенстве `id` — по `username` и так далее. Поэтому порядок полей в классе имеет значение. Параметр `order=True` требует `eq=True` (это значение по умолчанию).

In [ ]:
@dataclass(order=True)
class OrderedUser:
    id: int
    username: str
    password: str
    email: str
    created_at: datetime


alice = OrderedUser(1, "alice", "secret123", "alice@example.com", registered_at)
bob = OrderedUser(2, "bob", "qwerty123", "bob@example.com", registered_at)
carol = OrderedUser(3, "carol", "pa55word1", "carol@example.com", registered_at)
sorted_users = sorted([carol, alice, bob])
print(alice < bob)                               # True
print([user.username for user in sorted_users])  # ['alice', 'bob', 'carol']

### `init=False`: собственный `__init__`

Сгенерированный `__init__` присваивает полям переданные значения как есть. Если при создании экземпляра нужна собственная логика (например, привести `username` и `email` к нижнему регистру и самостоятельно заполнить `created_at`), метод `__init__` можно написать вручную и отключить генерацию параметром `init=False`. Строковое представление и сравнение по значениям при этом по-прежнему генерируются по аннотациям.

In [ ]:
@dataclass(init=False)
class ManualUser:
    id: int
    username: str
    password: str
    email: str
    created_at: datetime

    def __init__(self, id: int, username: str, password: str, email: str) -> None:
        self.id = id
        self.username = username.strip().lower()
        self.password = password
        self.email = email.strip().lower()
        self.created_at = datetime.now(UTC)


manual_user = ManualUser(1, "  Alice ", "secret123", "Alice@Example.com")
print(manual_user.username)    # alice
print(manual_user.email)       # alice@example.com
print(manual_user.created_at)  # 2026-11-19 20:03:37.611077+00:00 -- текущее время UTC

Если `__init__` определен в теле класса, декоратор не перезаписывает его и при `init=True`. Явное `init=False` показывает читателю кода, что инициализация написана вручную намеренно.

Как правило, собственный `__init__` не требуется: в разделах 7 и 8 рассмотрим `default_factory` и `__post_init__`, которые решают те же задачи, сохраняя сгенерированный `__init__`.

Итоги раздела:

| Параметр | Значение по умолчанию | Что делает |
|---|---|---|
| `frozen=True` | `False` | запрещает изменение полей после создания, добавляет хеширование по значениям полей |
| `kw_only=True` | `False` | все параметры `__init__` принимаются только по имени |
| `order=True` | `False` | добавляет операторы `<`, `<=`, `>`, `>=` (сравнение по кортежу полей) |
| `init=False` | `True` | не генерирует `__init__` |

## 4. Датакласс — обычный класс

Декоратор `dataclass` не создает особый вид объектов. Он изменяет **объект класса**: добавляет в его пространство имен сгенерированные функции (в том числе `__init__`) и возвращает **тот же** объект класса. Это видно на примере обычного класса, к которому декоратор применяется вызовом:

In [ ]:
class Product:
    title: str


print("__init__" in Product.__dict__)  # False
decorated_product = dataclass(Product)
print(decorated_product is Product)    # True -- возвращен тот же объект класса
print("__init__" in Product.__dict__)  # True -- функция добавлена в класс

Поэтому все, что известно об обычных классах, применимо к датаклассам: в них можно определять методы экземпляров, методы класса ([`classmethod`](conspects/05-class-methods.md)) и статические методы ([`staticmethod`](conspects/04-static-methods.md)). Добавим пользователю поведение:

- `get_email_domain` и `get_masked_password` — методы экземпляра;
- `normalize_email` — статический метод: он не использует ни экземпляр, ни класс;
- `from_json` — метод класса, альтернативный конструктор: обычный конструктор принимает значения полей, а `from_json` создает экземпляр из другого представления данных — JSON-строки.

Метод `from_json` работает так: функция `json.loads` преобразует JSON-строку в словарь, `email` нормализуется статическим методом, а затем словарь распаковывается в аргументы конструктора: `cls(**user_data)`. В JSON нет типа для даты, поэтому `created_at` передается строкой в формате ISO 8601 и преобразуется в `datetime` методом `datetime.fromisoformat`.

В аннотации возвращаемого значения `from_json` имя класса записано строкой `"User"`: во время выполнения тела класса имя `User` еще не определено.

In [ ]:
@dataclass
class User:
    id: int
    username: str
    password: str
    email: str
    created_at: datetime

    def get_email_domain(self) -> str:
        return self.email.split("@")[1]

    def get_masked_password(self) -> str:
        return "*" * len(self.password)

    @staticmethod
    def normalize_email(email: str) -> str:
        return email.strip().lower()

    @classmethod
    def from_json(cls, json_string: str) -> "User":
        user_data = json.loads(json_string)
        user_data["email"] = cls.normalize_email(user_data["email"])
        user_data["created_at"] = datetime.fromisoformat(user_data["created_at"])
        return cls(**user_data)

In [ ]:
json_string = (
    '{"id": 1, "username": "alice", "password": "secret123", '
    '"email": "  Alice@Example.COM ", "created_at": "2026-11-19T10:30:00"}'
)
user = User.from_json(json_string)
print(user.email)                  # alice@example.com
print(user.created_at)             # 2026-11-19 10:30:00
print(user.get_email_domain())     # example.com
print(user.get_masked_password())  # *********
print(isinstance(user, User))      # True

## 5. Атрибуты класса и `ClassVar`

Данные, общие для всех пользователей (минимальная длина пароля, имя таблицы в базе данных), хранятся в атрибутах класса. В датаклассе есть особенность: **любой атрибут с аннотацией становится полем**, даже если у него есть значение.

In [ ]:
@dataclass
class UserWithTable:
    id: int
    table_name: str = "users"


print(UserWithTable(1))

Атрибут `table_name` стал полем: он принимается в `__init__`, попадает в строковое представление экземпляра и в сравнение, и у каждого экземпляра хранится его собственная копия значения. Чтобы объявить атрибут класса, аннотацию записывают с типом `ClassVar` из модуля `typing`. Такие атрибуты датакласс игнорирует: они не становятся полями.

In [ ]:
@dataclass
class User:
    MIN_PASSWORD_LENGTH: ClassVar[int] = 8
    TABLE_NAME: ClassVar[str] = "users"
    id: int
    username: str
    password: str
    email: str
    created_at: datetime


user = User(1, "alice", "secret123", "alice@example.com", registered_at)
print(User.TABLE_NAME)  # users
print(user.TABLE_NAME)  # users -- атрибут класса виден через экземпляр

Атрибут **без аннотации** также остается обычным атрибутом класса, а не полем:

In [ ]:
@dataclass
class UserWithCurrency:
    id: int
    currency = "USD"


print(UserWithCurrency(1))        # UserWithCurrency(id=1)
print(UserWithCurrency.currency)  # USD

Аннотация `ClassVar` явно показывает, что значение принадлежит классу, и позволяет статическим анализаторам предупредить о попытке изменить его через экземпляр.

## 6. `__slots__` и параметр `slots=True`

### Где хранятся атрибуты экземпляра

Атрибуты экземпляра обычного класса, в том числе датакласса, хранятся в словаре `__dict__` этого экземпляра.

In [ ]:
print(list(user.__dict__))

У такого подхода два следствия:

1. **Расход памяти.** Для каждого экземпляра создается отдельный словарь. Если в программе миллионы однотипных записей, суммарный объем этих словарей становится заметным.
2. **Атрибуты можно добавлять динамически.** Опечатка в имени атрибута при присваивании не вызывает ошибку, а молча создает новый атрибут:

In [ ]:
user.emial = "alice@shop.example"  # опечатка в имени атрибута
print(user.email)                  # alice@example.com -- email не изменился
print(list(user.__dict__))

### `__slots__`

Атрибут класса `__slots__` содержит имена атрибутов, которые разрешено иметь экземплярам. Если он определен, Python не создает для экземпляра `__dict__`, а резервирует фиксированные места под перечисленные атрибуты прямо в структуре экземпляра.

In [ ]:
class SlottedUser:
    __slots__ = ("email", "id", "username")

    def __init__(self, id: int, username: str, email: str) -> None:
        self.id = id
        self.username = username
        self.email = email


slotted_user = SlottedUser(1, "alice", "alice@example.com")
print(hasattr(slotted_user, "__dict__"))  # False

try:
    slotted_user.emial = "alice@shop.example"

except AttributeError as error:
    print(error)

Таким образом, `__slots__` решает обе проблемы: экземпляр занимает меньше памяти, а присваивание атрибута с опечаткой в имени возбуждает `AttributeError`.

Вручную писать `__slots__` для датакласса неудобно: имена полей приходится дублировать, а значения по умолчанию, которые являются атрибутами класса, вызывают конфликт со `__slots__`. Поэтому в декоратор добавлен параметр **`slots=True`**: он сам формирует `__slots__` по полям датакласса.

Сравним расход памяти двух версий класса пользователя. Модуль `tracemalloc` отслеживает память, выделенную программе Python; функция `tracemalloc.get_traced_memory()` возвращает пару (текущий объем, пиковый объем) в байтах. Функция `measure_memory` создает заданное число экземпляров и возвращает объем занятой ими памяти.

In [ ]:
@dataclass
class DictUser:
    id: int
    username: str
    password: str
    email: str
    created_at: datetime


@dataclass(slots=True)
class SlotsUser:
    id: int
    username: str
    password: str
    email: str
    created_at: datetime


def measure_memory(user_class: type, users_count: int) -> int:
    tracemalloc.start()
    _ = [
        user_class(i, "alice", "secret123", "alice@example.com", registered_at)
        for i in range(users_count)
    ]
    current_size, _ = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return current_size


dict_size = measure_memory(DictUser, 100_000)
slots_size = measure_memory(SlotsUser, 100_000)
print(f"without slots: {dict_size // 1024} KiB")
print(f"with slots:    {slots_size // 1024} KiB")

Значения зависят от версии Python и платформы, но экономия за счет отсутствия `__dict__` заметна уже на ста тысячах экземпляров. Датакласс со `slots=True` ведет себя как класс с ручным `__slots__`:

In [ ]:
slots_user = SlotsUser(1, "alice", "secret123", "alice@example.com", registered_at)
print(SlotsUser.__slots__)
# ('id', 'username', 'password', 'email', 'created_at')
print(hasattr(slots_user, "__dict__"))  # False

try:
    slots_user.emial = "alice@shop.example"

except AttributeError as error:
    print(error)

Особенность параметра `slots=True`: атрибут `__slots__` должен быть задан **до** создания объекта класса, поэтому декоратор не изменяет исходный класс (как в разделе 4), а создает и возвращает **новый** класс:

In [ ]:
class Marker:
    value: int


slotted_marker = dataclass(slots=True)(Marker)
print(slotted_marker is Marker)  # False -- создан новый объект класса

Используйте `slots=True`, когда создается много экземпляров или нужно запретить добавление лишних атрибутов. Если экземплярам необходимо динамически добавлять атрибуты, `__slots__` не подходит.

## 7. Функция `field` и значения по умолчанию

### Значения по умолчанию для неизменяемых типов

Значение по умолчанию задается присваиванием в теле класса, как в параметрах функции. Для неизменяемых типов (`int`, `str`, `bool`, `tuple`, `datetime`) это безопасно.

In [ ]:
@dataclass
class User:
    id: int
    username: str
    password: str
    email: str
    is_active: bool = True


print(User(1, "alice", "secret123", "alice@example.com"))

Как и в параметрах функции, поле без значения по умолчанию не может следовать за полем со значением:

In [ ]:
try:
    @dataclass
    class BrokenUser:
        id: int
        is_active: bool = True
        username: str

except TypeError as error:
    print(error)

Решения: расположить поля с значениями по умолчанию в конце класса либо сделать поля только именованными (`kw_only=True` для класса или `field(kw_only=True)` для отдельного поля).

### Изменяемые значения по умолчанию

В [конспекте об изменяемых значениях по умолчанию](../lesson07/conspects/09-mutable-default-values.md) показано, что список, указанный как значение по умолчанию параметра функции, создается один раз и становится общим для всех вызовов. Тот же риск есть у полей датакласса: единственный список был бы общим для всех экземпляров. Датакласс не допускает такое значение и возбуждает ошибку при определении класса:

In [ ]:
try:
    @dataclass
    class CartUser:
        id: int
        cart: list[str] = []

except ValueError as error:
    print(error)

Для изменяемых значений используется функция `field` с параметром **`default_factory`**: ей передается вызываемый объект без аргументов, который вызывается **при создании каждого экземпляра**, а его результат становится значением поля. Вызов `list()` возвращает новый пустой список, поэтому у каждого экземпляра список свой.

In [ ]:
@dataclass
class User:
    id: int
    username: str
    password: str
    email: str
    cart: list[str] = field(default_factory=list)


first_user = User(1, "alice", "secret123", "alice@example.com")
second_user = User(2, "bob", "qwerty123", "bob@example.com")
first_user.cart.append("keyboard")
print(first_user.cart)                      # ['keyboard']
print(second_user.cart)                     # []
print(first_user.cart is second_user.cart)  # False

Параметры `default` и `default_factory` взаимоисключающие: указывать можно только один из них.

### `default_factory` для даты создания

Дату создания записи нужно вычислять в момент создания экземпляра. Типичная ошибка — записать вызов `datetime.now()` в качестве значения по умолчанию. Выражение вычисляется **один раз**, при выполнении тела класса, и значение становится общим для всех экземпляров. Датакласс ошибку не возбуждает, так как `datetime` неизменяем:

In [ ]:
@dataclass
class WrongUser:
    id: int
    created_at: datetime = datetime.now(UTC)


first_wrong_user = WrongUser(1)
time.sleep(0.01)
second_wrong_user = WrongUser(2)
print(first_wrong_user.created_at == second_wrong_user.created_at)
# True -- время зафиксировано при определении класса

Правильное решение — передать в `default_factory` вызываемый объект, который возвращает текущее время: он будет вызываться при создании каждого экземпляра. Здесь для этого подходит лямбда-функция без параметров `lambda: datetime.now(UTC)`. Константа `UTC` из модуля `datetime` задает часовой пояс UTC: время создания записей принято хранить в UTC.

In [ ]:
@dataclass
class User:
    id: int
    username: str
    password: str
    email: str
    created_at: datetime = field(default_factory=lambda: datetime.now(UTC))


first_user = User(1, "alice", "secret123", "alice@example.com")
time.sleep(0.01)
second_user = User(2, "bob", "qwerty123", "bob@example.com")
print(first_user.created_at < second_user.created_at)
# True -- время вычислено при создании каждого экземпляра

Фабрика вызывается только тогда, когда значение не передано. Поэтому при восстановлении существующей записи (например, из базы данных) дату создания можно указать явно:

In [ ]:
restored_user = User(3, "carol", "pa55word1", "carol@example.com", registered_at)
print(restored_user.created_at)  # 2026-11-19 10:30:00

Функция `field` принимает и другие параметры, которые управляют поведением отдельного поля: например, можно исключить поле из `__init__` или из сравнения. Подробнее о них можно прочитать в [документации](https://docs.python.org/3/library/dataclasses.html#dataclasses.field).

## 8. Метод `__post_init__`: проверка и нормализация значений

Сгенерированный `__init__` присваивает значения полям и больше ничего не делает: он не проверяет типы и не контролирует допустимость значений. Для дополнительных действий датакласс поддерживает метод **`__post_init__`**. Сгенерированный `__init__` вызывает его автоматически **после того, как все поля присвоены**. Метод принимает только `self` и ничего не возвращает.

В `__post_init__` обычно выполняют:

- **проверку значений**: при некорректных данных возбуждают исключение, и экземпляр не создается;
- **нормализацию**: приводят значения к единому виду (удаляют пробелы, переводят `email` в нижний регистр);
- вычисление значений, которые зависят от других полей.

In [ ]:
@dataclass
class User:
    MIN_PASSWORD_LENGTH: ClassVar[int] = 8

    id: int
    username: str
    password: str
    email: str
    created_at: datetime = field(default_factory=lambda: datetime.now(UTC))

    def __post_init__(self) -> None:
        self.username = self.username.strip()
        self.email = self.email.strip().lower()

        if not isinstance(self.id, int):
            raise TypeError(f"id must be int, got {type(self.id).__name__}")

        if self.id <= 0:
            raise ValueError("id must be positive")

        if not self.username:
            raise ValueError("username must not be empty")

        if len(self.password) < self.MIN_PASSWORD_LENGTH:
            raise ValueError(
                f"password must contain at least {self.MIN_PASSWORD_LENGTH} characters"
            )

        if "@" not in self.email:
            raise ValueError(f"invalid email: {self.email}")

Нормализация выполняется при каждом создании экземпляра:

In [ ]:
user = User(1, "  alice ", "secret123", "  Alice@Example.COM ")
print(user.username)  # alice
print(user.email)     # alice@example.com

Функция `try_create_user` пробует создать пользователя и печатает результат: так проверим все ветки валидации без повторяющихся блоков `try-except`.

In [ ]:
def try_create_user(**user_data: Any) -> None:
    try:
        User(**user_data)

    except (TypeError, ValueError) as error:
        print(f"{type(error).__name__}: {error}")

    else:
        print("created")


valid_data = {
    "id": 1,
    "username": "alice",
    "password": "secret123",
    "email": "alice@example.com",
}
invalid_changes = [
    {"id": "abc"},
    {"id": 0},
    {"username": "   "},
    {"password": "short"},
    {"email": "alice.example.com"},
]
try_create_user(**valid_data)

for changes in invalid_changes:
    try_create_user(**(valid_data | changes))

Замечание: в датаклассе с `frozen=True` присваивание атрибутов в `__post_init__` невозможно (оно возбуждает `FrozenInstanceError`), поэтому там допустима только проверка значений, но не нормализация.

## 9. Преобразование датакласса: `asdict` и `astuple`

Данные пользователя нужно не только хранить в памяти, но и передавать наружу: сохранять в файл или базу данных, отправлять в качестве ответа API, сравнивать в тестах. Для этого нужно преобразовать экземпляр в стандартные структуры данных. Модуль `dataclasses` предоставляет две функции: `asdict` и `astuple`.

### `asdict`: экземпляр в словарь

Функция `asdict(obj)` возвращает словарь `{имя поля: значение}`.

In [ ]:
user = User(1, "alice", "secret123", "alice@example.com", registered_at)
user_data = asdict(user)
print(user_data)
# {'id': 1, 'username': 'alice', 'password': 'secret123', 'email': 'alice@example.com',
#  'created_at': datetime.datetime(2026, 11, 19, 10, 30)}

Функция обрабатывает значения **рекурсивно**: вложенные датаклассы превращаются во вложенные словари, а списки, кортежи и словари копируются. Изменение результата не затрагивает исходный объект.

In [ ]:
@dataclass
class Order:
    id: int
    user: User
    items: list[str]


order = Order(100, user, ["keyboard", "mouse"])
order_data = asdict(order)
print(type(order_data["user"]))         # <class 'dict'>
print(order_data["user"]["username"])   # alice
order_data["items"].append("monitor")
print(order.items)  # ['keyboard', 'mouse'] -- исходный список не изменился

Основное применение `asdict` — сериализация в форматы, которые работают со словарями, например в JSON. Значение `datetime` модуль `json` самостоятельно не сериализует, поэтому его преобразуют в строку методом `isoformat()`:

In [ ]:
try:
    json.dumps(user_data)

except TypeError as error:
    print(error)  # Object of type datetime is not JSON serializable

In [ ]:
user_data["created_at"] = user_data["created_at"].isoformat()
json_text = json.dumps(user_data)
print(json_text)

### `astuple`: экземпляр в кортеж

Функция `astuple(obj)` возвращает кортеж значений полей в порядке объявления; вложенные датаклассы и коллекции обрабатываются рекурсивно, как в `asdict`. Кортеж удобен, когда нужны значения без имен: для распаковки, параметров SQL-запроса, строк таблицы или файла CSV.

In [ ]:
print(astuple(user))
user_id, username, password, email, created_at = astuple(user)
print(username, email)  # alice alice@example.com

Итоги раздела:

| Функция | Результат | Типичное применение |
|---|---|---|
| `asdict(obj)` | словарь `{имя: значение}` | JSON, ответы API; обратно: `Class(**data)` |
| `astuple(obj)` | кортеж значений | распаковка, параметры SQL-запросов, строки таблиц и CSV |